<div style='background:linear-gradient(135deg,#2E7D50,#1B5E38);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>🏗️ Bonus — Mini ML Project: Salary Prediction End-to-End</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 120 mins | Level: Intermediate</p>
<p style='margin:4px 0 0'><em>Capstone for Chapters 4–7 · No sklearn · Python + NumPy + Pandas only</em></p>
</div>

## 📋 What You Will Build

In this capstone project you will build a **salary predictor** using only the tools from Chapters 4–7:

| Phase | What You Do | Tools Used |
|-------|------------|------------|
| 1 — Explore | EDA: understand the data | Pandas |
| 2 — Preprocess | Clean, encode, normalise, split | Pandas + NumPy |
| 3 — Model | Gradient descent linear regression | NumPy |
| 4 — Analyse | Interpret weights, compare predictions | NumPy + Pandas |

**Constraint:** No sklearn. No scipy. No statsmodels. This is intentional — you are building intuition.

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports — all imports go here
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
np.random.seed(42)
pd.set_option('display.float_format', '{:.2f}'.format)
print('✅ Libraries loaded. Starting project...')

## 📋 Dataset Generation

We generate a synthetic dataset of 500 Indian tech employees. Salary is designed to have a realistic signal:
- More experience → higher salary
- Higher education → higher salary
- Metro cities (tier 1) → higher salary
- High skills score → higher salary
- Random noise → ±₹80,000 to simulate real-world messiness

In [ ]:
# ─────────────────────────────────────────────────────────────
# Dataset Generation — run this cell once
# ─────────────────────────────────────────────────────────────
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'experience':   np.random.randint(0, 20, n),
    'education':    np.random.choice([0, 1, 2], n),    # 0=UG, 1=PG, 2=PhD
    'city_tier':    np.random.choice([1, 2, 3], n),    # 1=Metro, 2=Tier2, 3=Tier3
    'skills_score': np.random.randint(40, 100, n),
    'dept':         np.random.choice(['Eng', 'Sales', 'HR', 'Finance'], n),
})

# Salary with realistic signal + noise
df['salary'] = (
    300000
    + df['experience']  * 80000
    + df['education']   * 120000
    + (4 - df['city_tier']) * 50000
    + df['skills_score'] * 2000
    + np.random.normal(0, 80000, n)
).astype(int)

print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print(f'\nSalary range: ₹{df.salary.min():,} – ₹{df.salary.max():,}')

---
## Section 1 — Data Exploration

Before building any model, understand your data. Every minute here saves an hour of debugging later.

---

> 🎯 **Task 1a — EDA: Shape, Types, Missing Values**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 1a: EDA — Basic Inspection
# ─────────────────────────────────────────────────────────────

# Print shape, dtypes, missing value counts, and describe()
# YOUR CODE HERE

In [ ]:
# @title ✅ Solution — Task 1a (click to expand)
# @markdown > Run this cell to reveal the solution

print('Shape:', df.shape)
print('\nData types:\n', df.dtypes)
print('\nMissing values:\n', df.isnull().sum())
print('\nDescriptive statistics:')
print(df.describe())

> 🎯 **Task 1b — Distribution: Salary by Department, Education, City Tier**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 1b: Salary Distributions
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: Average salary by department
# YOUR CODE HERE

# Plot 2: Average salary by education level
# YOUR CODE HERE

# Plot 3: Average salary by city tier
# YOUR CODE HERE

plt.tight_layout()
plt.show()

In [ ]:
# @title ✅ Solution — Task 1b (click to expand)
# @markdown > Run this cell to reveal the solution

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
edu_labels = {0: 'UG', 1: 'PG', 2: 'PhD'}
tier_labels = {1: 'Metro', 2: 'Tier 2', 3: 'Tier 3'}

# Plot 1: By department
dept_avg = df.groupby('dept')['salary'].mean().sort_values(ascending=False)
axes[0].bar(dept_avg.index, dept_avg.values / 100000, color='steelblue')
axes[0].set_title('Avg Salary by Department')
axes[0].set_ylabel('Salary (LPA)')

# Plot 2: By education
edu_avg = df.groupby('education')['salary'].mean()
axes[1].bar([edu_labels[i] for i in edu_avg.index], edu_avg.values / 100000, color='darkorange')
axes[1].set_title('Avg Salary by Education')

# Plot 3: By city tier
tier_avg = df.groupby('city_tier')['salary'].mean()
axes[2].bar([tier_labels[i] for i in tier_avg.index], tier_avg.values / 100000, color='seagreen')
axes[2].set_title('Avg Salary by City Tier')

for ax in axes: ax.set_xlabel('')
plt.tight_layout()
plt.show()
print('Notice: Metro earns more than Tier 3 — the city_tier signal is clear in the data.')

> 🎯 **Task 1c — Correlation: Which features correlate most with salary?**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 1c: Correlation Analysis
# ─────────────────────────────────────────────────────────────

# Compute and display correlation matrix for numeric features
# Highlight the correlation with 'salary'
# YOUR CODE HERE

In [ ]:
# @title ✅ Solution — Task 1c (click to expand)
# @markdown > Run this cell to reveal the solution

numeric_df = df.select_dtypes(include='number')
corr = numeric_df.corr()

print('Correlation with salary:')
print(corr['salary'].sort_values(ascending=False))

plt.figure(figsize=(8, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()
print('\nKey insight: experience and skills_score are most correlated with salary.')
print('city_tier is NEGATIVELY correlated — because tier 1 (Metro) = lower number = higher salary.')

---
## Section 2 — Data Preprocessing

Clean data is the foundation of any working model. Garbage in → garbage out.

---

> 🎯 **Task 2a — One-Hot Encode 'dept' using pd.get_dummies()**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 2a: One-Hot Encoding
# ─────────────────────────────────────────────────────────────

# One-hot encode 'dept' and drop the original column
# Use drop_first=True to avoid the dummy variable trap
df_encoded = # YOUR CODE HERE
print('Columns after encoding:', df_encoded.columns.tolist())
print('Shape:', df_encoded.shape)

In [ ]:
# @title ✅ Solution — Task 2a (click to expand)
# @markdown > Run this cell to reveal the solution

# pd.get_dummies converts a categorical column to binary columns
# drop_first=True removes one category to avoid multicollinearity
df_encoded = pd.get_dummies(df, columns=['dept'], drop_first=True)
print('Columns after encoding:', df_encoded.columns.tolist())
print('Shape:', df_encoded.shape)
print('\nNew dept columns:')
dept_cols = [c for c in df_encoded.columns if c.startswith('dept_')]
print(df_encoded[dept_cols].head())

> 🎯 **Task 2b — Normalise all numeric features to [0, 1] and split 80/20 train/test**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 2b: Normalise and Split
# ─────────────────────────────────────────────────────────────

# Separate features (X) and target (y)
feature_cols = [c for c in df_encoded.columns if c != 'salary']
X = df_encoded[feature_cols].values.astype(float)
y = df_encoded['salary'].values.astype(float)

# Normalise each feature column to [0, 1] using NumPy broadcasting
X_min = # YOUR CODE HERE
X_max = # YOUR CODE HERE
X_norm = # YOUR CODE HERE

# 80/20 train/test split using NumPy permutation (no sklearn)
np.random.seed(42)
indices = np.random.permutation(len(X_norm))
split   = int(len(X_norm) * 0.8)
X_train, X_test = # YOUR CODE HERE
y_train, y_test = # YOUR CODE HERE

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  y_test: {y_test.shape}')
print(f'Feature columns: {feature_cols}')

assert X_train.shape[0] == 400, f'Expected 400 training samples, got {X_train.shape[0]}'
assert X_test.shape[0]  == 100, f'Expected 100 test samples, got {X_test.shape[0]}'
assert abs(X_norm.max() - 1.0) < 1e-9, 'Max of normalised features should be 1.0'
print('\n✅ Preprocessing complete!')

In [ ]:
# @title ✅ Solution — Task 2b (click to expand)
# @markdown > Run this cell to reveal the solution

feature_cols = [c for c in df_encoded.columns if c != 'salary']
X = df_encoded[feature_cols].values.astype(float)
y = df_encoded['salary'].values.astype(float)

# Min-Max normalisation using broadcasting — axis=0 means per column
X_min  = X.min(axis=0)
X_max  = X.max(axis=0)
X_norm = (X - X_min) / (X_max - X_min + 1e-8)  # +1e-8 avoids division by zero

# Shuffle indices, then slice at 80% mark
np.random.seed(42)
indices = np.random.permutation(len(X_norm))
split   = int(len(X_norm) * 0.8)
X_train, X_test = X_norm[indices[:split]], X_norm[indices[split:]]
y_train, y_test = y[indices[:split]],      y[indices[split:]]

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'Feature columns: {feature_cols}')

assert X_train.shape[0] == 400
assert X_test.shape[0]  == 100
print('\n✅ Solution verified! This is exactly what train_test_split() does in sklearn.')

---
## Section 3 — Linear Regression from Scratch (Gradient Descent)

We will train a linear model using gradient descent — the same algorithm that trains neural networks.

**The Model:**
$\hat{y} = X \cdot W + b$

**The Loss:**
$\text{MSE} = \frac{1}{n} \sum (y_i - \hat{y}_i)^2$

**The Gradients:**
$\frac{\partial \text{MSE}}{\partial W} = \frac{2}{n} X^T (X W - y)$

**The Update:**
$W \leftarrow W - \alpha \cdot \nabla W$

---

> 🎯 **Task 3a — Implement Gradient Descent Linear Regression**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 3a: Gradient Descent Linear Regression
# ─────────────────────────────────────────────────────────────

def train_linear_regression(X, y, learning_rate=0.01, epochs=1000):
    """
    Train a linear regression model using gradient descent.
    Returns weights W and bias b after training.
    """
    n_samples, n_features = X.shape

    # Initialise weights and bias to zero
    W = np.zeros(n_features)
    b = 0.0

    loss_history = []

    for epoch in range(epochs):
        # Forward pass: compute predictions
        y_hat = # YOUR CODE HERE  (X @ W + b)

        # Compute MSE loss
        loss = # YOUR CODE HERE

        # Compute gradients
        dW = # YOUR CODE HERE   ((2/n) * X.T @ (y_hat - y))
        db = # YOUR CODE HERE   ((2/n) * np.sum(y_hat - y))

        # Update weights
        W  -= # YOUR CODE HERE
        b  -= # YOUR CODE HERE

        loss_history.append(loss)
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1:4d}/{epochs} | MSE: {loss:.2e}')

    return W, b, loss_history

# Normalise salary target too for stable gradient descent
y_mean, y_std = y_train.mean(), y_train.std()
y_train_norm  = (y_train - y_mean) / y_std
y_test_norm   = (y_test  - y_mean) / y_std

W, b, history = train_linear_regression(X_train, y_train_norm, learning_rate=0.1, epochs=1000)

In [ ]:
# @title ✅ Solution — Task 3a (click to expand)
# @markdown > Run this cell to reveal the solution

def train_linear_regression(X, y, learning_rate=0.01, epochs=1000):
    """Gradient descent linear regression from scratch."""
    n_samples, n_features = X.shape
    W = np.zeros(n_features)
    b = 0.0
    loss_history = []

    for epoch in range(epochs):
        y_hat = X @ W + b                               # forward pass
        loss  = np.mean((y - y_hat) ** 2)               # MSE loss
        dW    = (2 / n_samples) * X.T @ (y_hat - y)    # gradient w.r.t W
        db    = (2 / n_samples) * np.sum(y_hat - y)    # gradient w.r.t b
        W    -= learning_rate * dW                      # weight update
        b    -= learning_rate * db                      # bias update
        loss_history.append(loss)
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1:4d}/{epochs} | MSE: {loss:.4f}')

    return W, b, loss_history

y_mean, y_std = y_train.mean(), y_train.std()
y_train_norm  = (y_train - y_mean) / y_std
y_test_norm   = (y_test  - y_mean) / y_std

W, b, history = train_linear_regression(X_train, y_train_norm, learning_rate=0.1, epochs=1000)
print('\n✅ Training complete!')

> 🎯 **Task 3b — Predict on Test Set, Compute MSE and R² Score**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 3b: Evaluation
# ─────────────────────────────────────────────────────────────

# Predict on test set (in normalised space), then convert back to INR
y_pred_norm = # YOUR CODE HERE
y_pred      = # YOUR CODE HERE  (de-normalise: pred * y_std + y_mean)

# Compute MSE and R² on the original salary scale
mse = # YOUR CODE HERE
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2  = # YOUR CODE HERE  (1 - ss_res/ss_tot)

print(f'Test MSE : {mse:.2e}')
print(f'Test RMSE: ₹{np.sqrt(mse):,.0f}')
print(f'R² Score : {r2:.4f}')

# Plot learning curve
plt.figure(figsize=(10, 4))
plt.plot(history)
plt.title('Training Loss (MSE) over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# @title ✅ Solution — Task 3b (click to expand)
# @markdown > Run this cell to reveal the solution

y_pred_norm = X_test @ W + b
y_pred      = y_pred_norm * y_std + y_mean      # de-normalise to INR

mse    = np.mean((y_test - y_pred) ** 2)
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2     = 1 - (ss_res / ss_tot)

print(f'Test MSE : {mse:.2e}')
print(f'Test RMSE: ₹{np.sqrt(mse):,.0f}  ← average prediction error in rupees')
print(f'R² Score : {r2:.4f}  ← fraction of salary variance explained by the model')

plt.figure(figsize=(10, 4))
plt.plot(history)
plt.title('Training Loss Curve — Loss decreasing = model is learning')
plt.xlabel('Epoch')
plt.ylabel('MSE (normalised scale)')
plt.grid(True, alpha=0.3)
plt.show()
print('\n✅ The loss curve should show rapid decrease then plateau — gradient descent working correctly.')

---
## Section 4 — Analysis
---

> 🎯 **Task 4a — Print the 3 Most Important Features**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 4a: Feature Importance (by absolute weight)
# ─────────────────────────────────────────────────────────────

# Pair feature names with weights, sort by absolute value
weights_df = pd.DataFrame({
    'feature': feature_cols,
    'weight':  W,
    'abs_weight': np.abs(W)
}).sort_values('abs_weight', ascending=False)

print('Top 5 most important features:')
print(weights_df.head())

plt.figure(figsize=(10, 5))
plt.bar(weights_df['feature'], weights_df['weight'],
        color=['green' if w > 0 else 'red' for w in weights_df['weight']])
plt.title('Feature Weights — Green = positive effect on salary')
plt.xticks(rotation=30, ha='right')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()
print('\nKey insight: experience and skills_score should have the largest positive weights.')

> 🎯 **Task 4b — Show 10 Predictions vs Actual Values**

In [ ]:
# ─────────────────────────────────────────────────────────────
# Task 4b: Predictions vs Actual
# ─────────────────────────────────────────────────────────────

comparison = pd.DataFrame({
    'Actual (₹)':    y_test[:10].astype(int),
    'Predicted (₹)': y_pred[:10].astype(int),
    'Error (₹)':     (y_test[:10] - y_pred[:10]).astype(int),
    'Error %':       ((y_test[:10] - y_pred[:10]) / y_test[:10] * 100).round(1)
})

print('First 10 predictions vs actual:')
print(comparison.to_string())

# Scatter plot: actual vs predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='steelblue', s=30)
mn, mx = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
plt.plot([mn, mx], [mn, mx], 'r--', label='Perfect prediction')
plt.xlabel('Actual Salary (₹)')
plt.ylabel('Predicted Salary (₹)')
plt.title(f'Actual vs Predicted  |  R² = {r2:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

---
## 💡 What sklearn Adds

You just wrote ~50 lines to build a linear regression model. Here is what `sklearn.LinearRegression` gives you on top:

```python
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
```

**What sklearn adds that our scratch version doesn't:**
- Uses the closed-form OLS solution (no gradient descent needed) → exact answer
- Handles numerical edge cases automatically
- Exposes `.coef_` and `.intercept_` neatly
- Integrates with `Pipeline`, `GridSearchCV`, `cross_val_score`
- Mathematically identical result — just with far less code

**Why we built it from scratch:** So you understand *what* sklearn is computing, not just *that* it works. When a model produces wrong results in production, you need to know what's inside the black box.

---

## 🏆 What You Built

<div style='background:#EDF4FB;border-left:6px solid #1B3A6B;padding:20px;border-radius:8px'>
<h3 style='color:#1B3A6B'>Project Summary</h3>
<table style='border-collapse:collapse;width:100%'>
<tr><td style='padding:6px'><strong>Lines of ML code written</strong></td><td>~120 lines (preprocessing + training + evaluation)</td></tr>
<tr><td style='padding:6px'><strong>ML workflow steps completed</strong></td><td>Steps 3–8 (EDA → preprocessing → feature engineering → model → train → evaluate)</td></tr>
<tr><td style='padding:6px'><strong>Chapter 4 concepts applied</strong></td><td>ML workflow, supervised regression, problem framing</td></tr>
<tr><td style='padding:6px'><strong>Chapter 5 concepts applied</strong></td><td>Functions, loops, dicts, list comprehensions</td></tr>
<tr><td style='padding:6px'><strong>Chapter 6 concepts applied</strong></td><td>NumPy vectorisation, broadcasting, linear algebra, MSE, gradient computation</td></tr>
<tr><td style='padding:6px'><strong>Chapter 7 concepts applied</strong></td><td>Pandas EDA, get_dummies, groupby, merge</td></tr>
</table>
</div>

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Bonus Capstone Complete!</h3>
<p>You just built a complete machine learning pipeline from scratch — without sklearn. That is not beginner work.</p>
<p>The gradient descent algorithm you implemented is the same one that trains:
<strong>Google Search ranking, Netflix recommendations, and GPT models</strong> — scaled to billions of parameters.</p>
<p>You now understand what's inside the black box. That understanding will make you a better engineer than someone who just calls <code>model.fit()</code>.</p>
<p><strong>What's next:</strong> Chapter 8 onwards introduces sklearn, which gives you all of this in 3 lines — but now you know why it works.</p>
</div>